# 05 · Time Series Anomaly Detection

## What this notebook covers
Time series anomalies require methods that respect temporal structure. This notebook covers LSTM-based reconstruction — the current production standard — and explains why earlier methods are useful as baselines but not primary detectors.

## Earlier methods — context

**STL decomposition** (Seasonal-Trend decomposition using Loess) is a classical algorithm that decomposes a time series into trend, seasonal, and residual components. Anomalies show up as large residuals. It is fast, interpretable, and still used as a baseline or for data exploration — but it assumes a regular seasonal pattern and struggles with multivariate data or complex temporal dependencies.

**ARIMA residuals**: fit an ARIMA model, flag points where |residual| exceeds a threshold. Works well for stationary, univariate series with simple autocorrelation structure. Like STL, it falls apart on multivariate or non-stationary data.

**LSTM reconstruction**: train an LSTM autoencoder on normal windows. High reconstruction error = anomaly. Handles multivariate, non-stationary, complex temporal patterns. This is the method to implement for production use cases.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

# Synthetic multivariate time series: 3 features, seasonal + trend + noise
n = 1000
t = np.arange(n)
feat1 = np.sin(2 * np.pi * t / 50) + 0.02 * t + np.random.randn(n) * 0.3
feat2 = np.cos(2 * np.pi * t / 30) + np.random.randn(n) * 0.2
feat3 = 0.5 * np.sin(2 * np.pi * t / 80) + np.random.randn(n) * 0.15
X_ts = np.column_stack([feat1, feat2, feat3])

# Inject anomalies at specific windows
anomaly_windows = [(150, 155), (400, 405), (700, 708), (850, 853)]
y_point = np.zeros(n, dtype=int)
for lo, hi in anomaly_windows:
    X_ts[lo:hi] += np.random.randn(hi - lo, 3) * 3.0
    y_point[lo:hi] = 1

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_ts)
print(f"Time series: {n} timesteps, 3 features, {y_point.sum()} anomalous points")

In [ ]:
WINDOW = 30

def make_windows(X, window_size):
    windows = []
    for i in range(len(X) - window_size):
        windows.append(X[i:i+window_size])
    return np.array(windows)

windows = make_windows(X_scaled, WINDOW)
# Use only non-anomalous windows for training
window_labels = np.array([y_point[i:i+WINDOW].max() for i in range(len(X_scaled) - WINDOW)])
train_windows = windows[window_labels == 0]
print(f"Training windows: {len(train_windows)}  |  All windows: {len(windows)}")

X_train_t = torch.FloatTensor(train_windows)
X_all_t   = torch.FloatTensor(windows)

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, latent_dim=8):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.enc_fc  = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc  = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, input_dim, batch_first=True)

    def forward(self, x):
        _, (h, _) = self.encoder(x)
        z = self.enc_fc(h[-1])                      # bottleneck
        dec_init = self.dec_fc(z).unsqueeze(0)
        # Repeat latent for each timestep
        dec_in = dec_init.squeeze(0).unsqueeze(1).repeat(1, x.size(1), 1)
        out, _ = self.decoder(dec_in, (dec_init, torch.zeros_like(dec_init)))
        return out

    def reconstruction_error(self, x_t):
        with torch.no_grad():
            recon = self.forward(x_t)
            return ((x_t - recon) ** 2).mean(dim=(1, 2)).numpy()

model = LSTMAutoencoder(input_dim=3)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X_train_t), batch_size=32, shuffle=True)

for epoch in range(50):
    for (b,) in loader:
        l = nn.MSELoss()(model(b), b)
        opt.zero_grad(); l.backward(); opt.step()

# Score all windows
errors = model.reconstruction_error(X_all_t)
auc = roc_auc_score(window_labels, errors)
print(f"LSTM AE ROC-AUC: {auc:.4f}")

In [ ]:
# Map window scores back to timesteps (max-pool)
point_scores = np.zeros(n)
counts = np.zeros(n)
for i, err in enumerate(errors):
    point_scores[i:i+WINDOW] = np.maximum(point_scores[i:i+WINDOW], err)

threshold = np.percentile(errors[window_labels==0], 95)
flagged = point_scores > threshold

fig, axes = plt.subplots(3, 1, figsize=(14, 8))
axes[0].plot(X_ts[:, 0], color="steelblue", lw=0.8, label="Feature 0")
for lo, hi in anomaly_windows:
    axes[0].axvspan(lo, hi, alpha=0.3, color="red")
axes[0].set_title("Time series (red = injected anomaly windows)")

axes[1].plot(point_scores, color="darkorange", lw=0.8, label="Reconstruction error")
axes[1].axhline(threshold, color="black", linestyle="--", label=f"Threshold={threshold:.3f}")
axes[1].set_title("LSTM reconstruction error (max-pooled)")
axes[1].legend(fontsize=8)

axes[2].fill_between(range(n), flagged.astype(float), color="tomato", alpha=0.8, label="Flagged")
axes[2].fill_between(range(n), y_point.astype(float), color="steelblue", alpha=0.4, label="True anomaly")
axes[2].set_title("Flags vs ground truth"); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{base}/05_timeseries_anomaly.png", dpi=120)
plt.show()